# 全生命周期产热 canonical

本 Notebook 合并旧版 `统一老化路径_多倍率全生命周期产热`、`MIC_1175Ah_0p25P统一老化路径_多倍率全生命周期产热` 和 `587Ah_lifecycle_heat_generation`。所有可变参数只在下方 `CONFIG` cell 修改；核心仿真、checkpoint/partial recovery、SOH 诊断、entropy calibration、趋势外推、中文 Excel 和绘图入口均在 `src` 中实现。


In [ ]:
CONFIG = {
    "cell": "587Ah",
    "run_mode": "smoke",  # smoke / study
    "output_name": "全生命周期产热",
    "conditions": [
        {
            "temperature_c": 25.0,
            "aging_p_rate": 0.5,
            "diagnostic_p_rates": [0.25, 0.5],
        },
        # MIC study 示例：25/35/45°C、0.25P 老化、0.125/0.167/0.25/0.5P 诊断
        # {"temperature_c": 35.0, "aging_p_rate": 0.25, "diagnostic_p_rates": [0.125, 0.167, 0.25, 0.5]},
        # {"temperature_c": 45.0, "aging_p_rate": 0.25, "diagnostic_p_rates": [0.125, 0.167, 0.25, 0.5]},
    ],
    "contact_resistances": [
        {"label": "R_low", "resistance_mohm": 0.056276},
        {"label": "R_high", "resistance_mohm": 0.061291},
        # MIC 示例：0.07018 / 0.0748 mOhm
    ],
    "soh_targets_pct": [100, 95, 90, 85, 80, 75, 70, 65, 60],
    "nominal_capacity_ah": 587.0,
    "nominal_voltage_v": 3.2,
    "charge_cutoff_v": 3.65,
    "discharge_cutoff_v": 2.50,
    "rest_minutes": 10.0,
    "period_minutes": 0.5,
    "solver_rtol": 1e-6,
    "solver_atol": 1e-6,
    "entropy": {
        "query": {
            "cell": "AI_virtual_cell",
            "kind": "raw",
            "format": "csv",
            "path_contains": "entropy_coefficient_fits_by_branch.csv",
            "require_unique": True,
        },
        "sample_label": "SOH95%",
        "equilibrium_charge_weight": 0.310139,
        "source_note": "转用314Ah SOH95%全电芯充放电支路dU/dT；作为587Ah/MIC临时先验时需在报告中标记",
    },
    "heat_correction": {"enabled": False, "scale": 1.0, "offset_w": 0.0},
    "collect_cycle_heat": False,
    "return_partial_on_error": True,
    "stop_at_lowest_diagnostic_soh": True,
    "keep_only_last_cycle_solution": True,
    "keep_only_last_capacity_check_solution": True,
    "var_pts": {"x_n": 5, "x_s": 5, "x_p": 5, "r_n": 20, "r_p": 20},
    "modes": {
        "smoke": {
            "soh_targets_pct": [100],
            "total_cycles": 1,
            "aging_t_factor": 1,
            "cycles_per_block": 1,
            "showprogress": False,
            "return_solutions": True,
        },
        "study": {
            "total_cycles": 20000,
            "aging_t_factor": 50,
            "cycles_per_block": 1,
            "showprogress": False,
            "return_solutions": True,
        },
    },
}

## 1. 环境与公共 API

Notebook 只负责 orchestration 和展示；不在本地定义核心函数。修改 `src` 后重跑本节以 reload 并重新绑定符号。


In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("science")
plt.rcParams["font.family"] = "Calibri, Microsoft YaHei"

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "BatteryProject" / "src").is_dir():
        PROJECT_ROOT = candidate / "BatteryProject"
        break
    if candidate.name == "BatteryProject" and (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
WORKSPACE_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook import setup_notebook
import src.workflows.lifecycle_heat as lifecycle_heat_module

importlib.reload(lifecycle_heat_module)
LifecycleHeatWorkflowSpec = lifecycle_heat_module.LifecycleHeatWorkflowSpec
build_feature_parity_table = lifecycle_heat_module.build_feature_parity_table
resolve_entropy_file = lifecycle_heat_module.resolve_entropy_file
run_lifecycle_heat_workflow = lifecycle_heat_module.run_lifecycle_heat_workflow

CTX = setup_notebook(cell=CONFIG["cell"], project_root=PROJECT_ROOT, style="science")
print("Project root:", PROJECT_ROOT)
print("Workspace root:", WORKSPACE_ROOT)

## 2. Feature parity 清单

以下矩阵是本 canonical 相对三份旧 Notebook 的功能并集检查。


In [ ]:
feature_parity = build_feature_parity_table()
display(feature_parity)

## 3. 配置校验与 DatasetQuery

熵热数据通过 `DatasetQuery` 查询 `datasets.json`；Notebook 不直接拼原始实验路径。多匹配会按配置报错，避免隐式选错数据。


In [ ]:
spec = LifecycleHeatWorkflowSpec.from_mapping(CONFIG)
entropy_file = resolve_entropy_file(spec, workspace_root=WORKSPACE_ROOT)
config_view = pd.DataFrame([
    ["电芯", spec.cell],
    ["运行模式", spec.run_mode],
    ["工况数", len(spec.conditions)],
    ["接触电阻数", len(spec.contact_resistances)],
    ["SOH目标", str(spec.soh_targets_pct)],
    ["熵热文件", str(entropy_file)],
], columns=["项目", "内容"])
display(config_view)

## 4. 运行 headless workflow

`smoke` 只跑最小检查点；`study` 才会执行完整多温度/多倍率/多接触电阻矩阵。输出写入 `BatteryProject/output/runs/lifecycle_heat/<run_id>/`。


In [ ]:
result = run_lifecycle_heat_workflow(
    spec,
    project_root=PROJECT_ROOT,
    workspace_root=WORKSPACE_ROOT,
)
print("Run dir:", result["context"].run_dir)
print("Metrics:", result["metrics_path"])
print("Artifacts:", result["artifact_manifest"])

## 5. SOH诊断与趋势外推

实际诊断点来自统一老化主线的 SOH crossing 分支；缺失低 SOH 点由 src 中的无约束趋势外推补齐，并保留 `数据来源` 标记。


In [ ]:
display(result["summary"])
display(result["detail"].head(20))

## 6. 老化机制与 SOH 诊断

保留 SEI、裂纹 SEI、析锂、负极 LAM、LLI 和孔隙率等诊断字段；机制占比只对独立项归一化，LLI 不重复计入分母。


In [ ]:
display(result["mechanism"].head(20))
display(result["mechanism_share"].head(20))

## 7. 逐 cycle 产热表

当 `CONFIG["collect_cycle_heat"] = True` 时，保留 587Ah 旧 Notebook 的 cycle/SOH 横坐标产热表入口；默认关闭以避免 smoke 过重。


In [ ]:
if result["cycle_heat"].empty:
    print("collect_cycle_heat=False，未生成逐cycle产热表。")
else:
    display(result["cycle_heat"].head())

## 8. 中文Excel、绘图和 artifact 链接

每个温度 × 老化倍率 × 接触电阻案例输出独立中文 Excel，并保存产热和机制图；artifact manifest 记录机器可读路径。


In [ ]:
for case_key, case in result["cases"].items():
    print(case_key)
    print("  Excel:", case["workbook"])
    for plot_path in case["plots"]:
        print("  Plot:", plot_path)

## 9. 后续复用

- 587Ah 默认可跑 0.25P/0.5P 工况；MIC 可把 `cell`、`nominal_capacity_ah`、`conditions` 和接触电阻切换到 MIC 配置。
- 多温度 study 只改 `CONFIG["conditions"]`，不要在后处理章节散落参数。
- entropy calibration 必须继续通过 DatasetQuery 选取 CSV；若换成 MIC 实测熵热，只改 CONFIG 中的 query 和 source_note。
- checkpoint/partial recovery、趋势外推和中文 Excel 均由 src API 维护，Notebook 不添加本地核心实现。
